## Mutation-Level Alternative Statistical Models
This notebook mirrors the main manuscript workflow: shared mutation-level curation first, then Essential/Nonessential/Combined dataset splits, then holdout and temporal reclassification evaluation in one pass.


## 1. Imports and Shared Mutation-Level Setup


In [7]:

import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import accuracy_score, auc, classification_report, confusion_matrix, f1_score, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

NONESSENTIAL_GENES = {"pncA", "gid", "ethA"}
DATASET_ORDER = ["Essential", "Nonessential", "Combined"]


def resolve_project_root() -> Path:
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd().parent.parent,
    ]
    for candidate in candidates:
        if (candidate / 'paper_release/source_data/derived_features/2021/2021_final_df.csv').exists():
            return candidate
    raise FileNotFoundError('Could not locate project root containing paper_release/source_data')


PROJECT_ROOT = resolve_project_root()
DATA_2021 = PROJECT_ROOT / 'paper_release/source_data/derived_features/2021/2021_final_df.csv'
DATA_2023 = PROJECT_ROOT / 'paper_release/source_data/derived_features/2023/2023_final_df.csv'
RESULTS_DIR = PROJECT_ROOT / 'Comparison_Model' / 'mutation_level_results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print('Resolved PROJECT_ROOT:', PROJECT_ROOT)

BINARY_MAP = {
    "1) Assoc w R": 1,
    "2) Assoc w R - Interim": 1,
    "4) Not assoc w R - Interim": 0,
    "5) Not assoc w R": 0,
}

CONFIDENCE_PRIORITY = {
    '1) Assoc w R': 0,
    '2) Assoc w R - Interim': 1,
    '4) Not assoc w R - Interim': 2,
    '5) Not assoc w R': 3,
    '3) Uncertain significance': 4,
}


def load_release_features(path):
    df = pd.read_csv(path).copy()
    rename_map = {
        'mutation_oneletter': 'one_letter_mutation',
        'Rosetta_fa_atr': 'fa_atr',
        'Rosetta_fa_rep': 'fa_rep',
        'Rosetta_fa_sol': 'fa_sol',
        'Rosetta_fa_elec': 'fa_elec',
        'Rosetta_fa_dun': 'fa_dun',
        'Rosetta_ddG': 'thermostability',
        'Prox_3D_zeroed': 'Proximity_to_R_Conferring',
        'LLR_score': 'llr_score',
        'freq_variant': 'frequency',
        'AAIndex_mut1': 'mut_AAIndex1',
        'AAIndex_mut2': 'mut_AAIndex2',
        'AAIndex_mut3': 'mut_AAIndex3',
        'AAIndex_mut4': 'mut_AAIndex4',
        'AAIndex_mut5': 'mut_AAIndex5',
        'AAIndex_mut6': 'mut_AAIndex6',
        'AAIndex_mut7': 'mut_AAIndex7',
        'AAIndex_mut8': 'mut_AAIndex8',
    }
    return df.rename(columns=rename_map)


def collapse_supervised_mutation_level(df):
    labeled = df[df['confidence'] != '3) Uncertain significance'].copy()
    labeled['_binary_confidence'] = labeled['confidence'].map(BINARY_MAP)
    labeled = labeled[labeled['_binary_confidence'].notna()].copy()
    pre_n = labeled.shape[0]
    labeled = labeled.drop_duplicates(subset=['gene', 'one_letter_mutation', '_binary_confidence']).copy()
    print(f'Collapsed supervised labeled set: {pre_n} -> {labeled.shape[0]} rows')
    return labeled.drop(columns=['_binary_confidence'])


def collapse_confidence_to_mutation_level(df, confidence_col='confidence'):
    rows = []
    for (gene, mut), sub in df.groupby(['gene', 'one_letter_mutation'], dropna=False):
        confs = [c for c in sub[confidence_col].dropna().tolist()]
        non_uncertain = [c for c in confs if c != '3) Uncertain significance']
        pool = non_uncertain if non_uncertain else confs
        chosen = sorted(pool, key=lambda c: CONFIDENCE_PRIORITY.get(c, 999))[0] if pool else np.nan
        first = sub.iloc[0].copy()
        first[confidence_col] = chosen
        rows.append(first)
    return pd.DataFrame(rows).reset_index(drop=True)


def merge_catalogs_mutation_level(df_2021, df_2023):
    df_2021_mut = collapse_confidence_to_mutation_level(df_2021, 'confidence')
    df_2023_mut = collapse_confidence_to_mutation_level(df_2023, 'confidence')
    merged_df = pd.merge(
        df_2021_mut,
        df_2023_mut,
        on=['gene', 'one_letter_mutation'],
        suffixes=('_2021', '_2023')
    )
    return merged_df.drop_duplicates()


def filter_by_dataset(df, dataset_name, gene_col='gene'):
    if dataset_name == 'Combined':
        return df.copy()
    if dataset_name == 'Nonessential':
        return df[df[gene_col].isin(NONESSENTIAL_GENES)].copy()
    if dataset_name == 'Essential':
        return df[~df[gene_col].isin(NONESSENTIAL_GENES)].copy()
    raise ValueError(f'Unknown dataset: {dataset_name}')


def summarize_binary_metrics(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) else np.nan
    specificity = tn / (tn + fp) if (tn + fp) else np.nan
    return {
        'Support S': int((y_true == 0).sum()),
        'Support R': int((y_true == 1).sum()),
        'Sensitivity': sensitivity,
        'Specificity': specificity,
        'Incorrect': int((y_true != y_pred).sum()),
        'Type I Error': fp / (fp + tn) if (fp + tn) else np.nan,
        'Type II Error': fn / (fn + tp) if (fn + tp) else np.nan,
    }


def build_reclassified_variant_sets(df_2021, df_2023, feature_columns):
    merged_df = merge_catalogs_mutation_level(df_2021, df_2023)
    unchanged_uncertain = merged_df[
        (merged_df['confidence_2021'] == '3) Uncertain significance') &
        (merged_df['confidence_2023'] == '3) Uncertain significance')
    ].drop_duplicates()

    reclassified_from_2021_uncertain = merged_df[
        (merged_df['confidence_2021'] == '3) Uncertain significance') &
        (merged_df['confidence_2023'] != '3) Uncertain significance')
    ].drop_duplicates(subset=['gene', 'one_letter_mutation', 'confidence_2021', 'confidence_2023']).copy()

    reclassified_from_2021_uncertain['confidence_2023'] = reclassified_from_2021_uncertain['confidence_2023'].map(BINARY_MAP)
    reclassified_from_2021_uncertain = reclassified_from_2021_uncertain[
        reclassified_from_2021_uncertain['confidence_2023'].notna()
    ].copy()

    uncertain_2021_df = df_2021[df_2021['confidence'] == '3) Uncertain significance'].copy()
    uncertain_2021_df = uncertain_2021_df.drop_duplicates(subset=['gene', 'one_letter_mutation'])
    reclassified_eval_df = uncertain_2021_df.merge(
        reclassified_from_2021_uncertain[['gene', 'one_letter_mutation', 'confidence_2023']].drop_duplicates(),
        on=['gene', 'one_letter_mutation'],
        how='inner'
    )
    reclassified_eval_df[feature_columns] = reclassified_eval_df[feature_columns].fillna(0)

    return merged_df, unchanged_uncertain, reclassified_from_2021_uncertain, reclassified_eval_df


catalog_data = load_release_features(DATA_2021)
catalog_data = catalog_data.drop_duplicates()
catalog_data = catalog_data[catalog_data['Prox_3D'].notna()].copy()
print('Catalog rows after Prox_3D filter:', catalog_data.shape[0])

full_label_data = collapse_supervised_mutation_level(catalog_data)
full_label_data['binary_confidence'] = full_label_data['confidence'].map(BINARY_MAP)
print('Labeled rows available for modeling:', full_label_data.shape[0])

raw_2021 = load_release_features(DATA_2021)
raw_2023 = load_release_features(DATA_2023)
raw_2021 = raw_2021[raw_2021['Prox_3D'].notna()].copy()
raw_2023 = raw_2023[raw_2023['Prox_3D'].notna()].copy()


Resolved PROJECT_ROOT: /project/pi_annagreen_umass_edu/mahbuba/all_projects/resistance_forecast
Catalog rows after Prox_3D filter: 4248
Collapsed supervised labeled set: 370 -> 345 rows
Labeled rows available for modeling: 345


## 2. Dataset Curation and Gene-Stratified Splits
The dataset split happens here, up front, so the rest of the notebook follows the same three-way flow as the main manuscript notebook.


In [8]:

numeric_columns = [
    'fa_atr', 'fa_rep', 'fa_sol', 'fa_elec', 'fa_dun', 'thermostability',
    'Proximity_to_R_Conferring',
    'mut_AAIndex1', 'mut_AAIndex2', 'mut_AAIndex3', 'mut_AAIndex4',
    'mut_AAIndex5', 'mut_AAIndex6', 'mut_AAIndex7', 'mut_AAIndex8',
    'LLR_dim33', 'LLR_dim46', 'LLR_dim62', 'LLR_dim70', 'LLR_dim124',
    'LLR_dim192', 'LLR_dim207', 'LLR_dim258', 'LLR_dim267', 'LLR_dim315',
    'frequency'
]

for col in numeric_columns:
    if col not in full_label_data.columns:
        raise KeyError(f'Missing expected GAQQ feature column: {col}')

print('GAQQ feature count:', len(numeric_columns))
print(numeric_columns)


GAQQ feature count: 26
['fa_atr', 'fa_rep', 'fa_sol', 'fa_elec', 'fa_dun', 'thermostability', 'Proximity_to_R_Conferring', 'mut_AAIndex1', 'mut_AAIndex2', 'mut_AAIndex3', 'mut_AAIndex4', 'mut_AAIndex5', 'mut_AAIndex6', 'mut_AAIndex7', 'mut_AAIndex8', 'LLR_dim33', 'LLR_dim46', 'LLR_dim62', 'LLR_dim70', 'LLR_dim124', 'LLR_dim192', 'LLR_dim207', 'LLR_dim258', 'LLR_dim267', 'LLR_dim315', 'frequency']


In [9]:

datasets = {}
for dataset_name in DATASET_ORDER:
    labeled_df = filter_by_dataset(full_label_data, dataset_name)
    df_2021_subset = filter_by_dataset(raw_2021, dataset_name)
    df_2023_subset = filter_by_dataset(raw_2023, dataset_name)
    merged_df, unchanged_uncertain, reclassified_from_2021_uncertain, reclassified_eval_df = build_reclassified_variant_sets(
        df_2021_subset, df_2023_subset, numeric_columns
    )
    datasets[dataset_name] = {
        'labeled_df': labeled_df,
        'df_2021': df_2021_subset,
        'df_2023': df_2023_subset,
        'merged_df': merged_df,
        'unchanged_uncertain': unchanged_uncertain,
        'reclassified_from_2021_uncertain': reclassified_from_2021_uncertain,
        'reclassified_eval_df': reclassified_eval_df,
    }

    print('Labeled rows:', labeled_df.shape[0])
    print('Temporal unchanged uncertain:', unchanged_uncertain.shape[0])
    print('Temporal reclassified variants:', reclassified_eval_df.shape[0])
    if not reclassified_eval_df.empty:
        print('Temporal class counts:', reclassified_eval_df['confidence_2023'].value_counts().to_dict())


Labeled rows: 159
Temporal unchanged uncertain: 2294
Temporal reclassified variants: 7
Temporal class counts: {1: 6, 0: 1}
Labeled rows: 186
Temporal unchanged uncertain: 820
Temporal reclassified variants: 55
Temporal class counts: {1: 51, 0: 4}
Labeled rows: 345
Temporal unchanged uncertain: 3114
Temporal reclassified variants: 62
Temporal class counts: {1: 57, 0: 5}


## 3. GAQQ Holdout and Temporal Reclassification Evaluation


In [12]:

import rpy2.robjects as ro
from rpy2.robjects import pandas2ri

pandas2ri.activate()
r = ro.r
R_SCRIPT = (PROJECT_ROOT / 'Comparison_Model' / 'est.twoclass.R').resolve()
r['source'](str(R_SCRIPT))
print(f'R script loaded successfully from {R_SCRIPT}.')

SLDA_EBIC = ro.globalenv['SLDA.EBIC']
classify = ro.globalenv['classify']


def run_gaqq_for_dataset(dataset_name, dataset_bundle):
    labeled_df = dataset_bundle['labeled_df'].copy()
    X_label = labeled_df[numeric_columns].fillna(0)
    y_label = labeled_df['confidence'].map(BINARY_MAP)

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_label)
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y_label, test_size=0.3, random_state=42
    )

    X1 = X_train[y_train == 1]
    X2 = X_train[y_train == 0]
    X3 = X_test[y_test == 1]
    X4 = X_test[y_test == 0]
    lambda1 = np.linspace(0.01, 5, 50)
    lambda2 = np.linspace(0.01, 5, 50)

    result = SLDA_EBIC(X1, X2, lambda1, lambda2)
    C_opt = np.array(result.rx2('C_opt'))
    delta_h_opt = result.rx2('delta_h_opt')
    C_opt_regularized = C_opt + 1e-8 * np.eye(C_opt.shape[0])
    C_opt_r = ro.r.matrix(C_opt_regularized, nrow=C_opt_regularized.shape[0], ncol=C_opt_regularized.shape[1])

    holdout_result = classify(X1, X2, X3, X4, C_opt_r, delta_h_opt)
    holdout_scores = np.array(holdout_result.rx2('y_hat'), dtype=float).reshape(-1)
    holdout_pred = 1 - np.array(holdout_result.rx2('est_label'), dtype=int).reshape(-1)
    holdout_metrics = summarize_binary_metrics(np.asarray(y_test), holdout_pred)
    holdout_auc = roc_auc_score(y_test, holdout_scores) if len(np.unique(y_test)) == 2 else np.nan

    reclassified_eval_df = dataset_bundle['reclassified_eval_df'].copy()
    X_reclassified = scaler.transform(reclassified_eval_df[numeric_columns].fillna(0))
    y_true_temporal = reclassified_eval_df['confidence_2023'].to_numpy()
    X3_temporal = X_reclassified[y_true_temporal == 1]
    X4_temporal = X_reclassified[y_true_temporal == 0]

    temporal_result = classify(X1, X2, X3_temporal, X4_temporal, C_opt_r, delta_h_opt)
    temporal_scores = np.array(temporal_result.rx2('y_hat'), dtype=float).reshape(-1)
    temporal_pred = 1 - np.array(temporal_result.rx2('est_label'), dtype=int).reshape(-1)
    temporal_metrics = summarize_binary_metrics(y_true_temporal, temporal_pred)
    temporal_auc = roc_auc_score(y_true_temporal, temporal_scores) if len(np.unique(y_true_temporal)) == 2 else np.nan

    return {
        'Dataset': dataset_name,
        'Model': 'GAQQ',
        'Labeled Samples': int(labeled_df.shape[0]),
        'Reclassified Variants': int(reclassified_eval_df.shape[0]),
        'Holdout AUC': holdout_auc,
        'Holdout Accuracy': accuracy_score(y_test, holdout_pred),
        'Holdout Sens (R)': holdout_metrics['Sensitivity'],
        'Holdout Spec (S)': holdout_metrics['Specificity'],
        'Holdout Support R': holdout_metrics['Support R'],
        'Holdout Support S': holdout_metrics['Support S'],
        'Temporal AUC': temporal_auc,
        'Temporal Accuracy': accuracy_score(y_true_temporal, temporal_pred),
        'Temporal F1 (R)': f1_score(y_true_temporal, temporal_pred, pos_label=1, zero_division=0),
        'Temporal Sens (R)': temporal_metrics['Sensitivity'],
        'Temporal Spec (S)': temporal_metrics['Specificity'],
        'Temporal Support R': temporal_metrics['Support R'],
        'Temporal Support S': temporal_metrics['Support S'],
        'Temporal Incorrect': temporal_metrics['Incorrect'],
        'Type I Error': temporal_metrics['Type I Error'],
        'Type II Error': temporal_metrics['Type II Error'],
    }


R script loaded successfully from /project/pi_annagreen_umass_edu/mahbuba/all_projects/resistance_forecast/Comparison_Model/est.twoclass.R.


In [13]:

gaqq_summary_df = pd.DataFrame([
    run_gaqq_for_dataset(dataset_name, datasets[dataset_name])
    for dataset_name in DATASET_ORDER
])

gaqq_summary_df.to_csv(RESULTS_DIR / 'gaqq_mutation_level_summary.csv', index=False)
gaqq_summary_df


,Dataset,Model,Labeled Samples,Reclassified Variants,Holdout AUC,Holdout Accuracy,Holdout Sens (R),Holdout Spec (S),Holdout Support R,Holdout Support S,Temporal AUC,Temporal Accuracy,Temporal F1 (R),Temporal Sens (R),Temporal Spec (S),Temporal Support R,Temporal Support S,Temporal Incorrect,Type I Error,Type II Error
0,Essential,GAQQ,159,7,0.491453,0.729167,0.871795,0.111111,39,9,0.416667,0.714286,0.833333,0.833333,0.0,6,1,2,1.0,0.166667
1,Nonessential,GAQQ,186,55,0.500000,0.875000,1.000000,0.000000,49,7,0.500000,0.927273,0.962264,1.000000,0.0,51,4,4,1.0,0.000000
2,Combined,GAQQ,345,62,0.495605,0.750000,0.873563,0.117647,87,17,0.473684,0.870968,0.931034,0.947368,0.0,57,5,8,1.0,0.052632
